# 07 — Read / Write

Reading from files (CSV/Parquet/JSON), writing back to IRIS, creating DataFrames from pandas/polars/lists, and the `irispark.io` module.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. `createDataFrame` from pandas / lists

In [ ]:
import pandas as pd

pdf = pd.DataFrame({"a": [1, 2, 3], "b": ["x", "y", "z"]})
df = session.createDataFrame(pdf)
df.show()

from_list = session.createDataFrame([(1, "a"), (2, "b")], ["id", "letra"])
from_list.show()

## 2. Write to IRIS — `saveAsTable` / `insertInto`

In [ ]:
df.write.saveAsTable("exemplo_tabela")
print(session.table("exemplo_tabela").show())

# insertInto appends; the writer defaults to mode 'error', so opt into append
session.createDataFrame(pd.DataFrame({"a": [4], "b": ["w"]})).write.mode("append").insertInto("exemplo_tabela")
session.table("exemplo_tabela").show()

## 3. Write to files — CSV / Parquet

In [ ]:
import tempfile, os

tmp = tempfile.mkdtemp()

df.write.csv(os.path.join(tmp, "out.csv"))
df.write.parquet(os.path.join(tmp, "out.parquet"))
print("wrote to", tmp)

## 4. Read from files — local

In [ ]:
csv_path = os.path.join(tmp, "out.csv")
parquet_path = os.path.join(tmp, "out.parquet")

session.read.csv(csv_path).show()
session.read.parquet(parquet_path).show()

## 5. Read via IRIS Foreign Tables — `foreign=True`

IRIS owns the file connection; rows are not copied into Python. The file lives in the repo's `data/` directory, which compose mounts into the IRIS container at `/irispark-data` — so we tell IRIS where *it* sees the folder via `server_path`.

In [ ]:
import os

csv_dir = "data/foreign_demo"
os.makedirs(csv_dir, exist_ok=True)
pdf.head(3).to_csv(os.path.join(csv_dir, "out.csv"), index=False)

try:
    session.read.csv(
        os.path.join(csv_dir, "out.csv"),
        foreign=True,
        server_path="/irispark-data/foreign_demo",
        options={"header": True},
    ).show()
except Exception as e:
    print("Foreign table read not available:", e)

## 6. `irispark.io` module

Module-level functions mirroring pandas-on-Spark I/O.

In [ ]:
from irispark.io import read_csv, from_pandas, read_sql

read_csv(csv_path, session=session).show()
from_pandas(pdf, session=session).show()
read_sql("SELECT * FROM vendas", session=session).show()

## 7. Cleanup

Drop the example table.

In [ ]:
session.sql('DROP TABLE IF EXISTS exemplo_tabela')
print("dropped exemplo_tabela")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")